# 01 - Exploratory Data Analysis

Goal: understand the data and confirm that next-day direction is genuinely hard. Run `python -m src.cli download --tickers AAPL` first.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import data as dmod
from src.utils import FIGURES_DIR, ensure_dirs

ensure_dirs()
df = dmod.load_dataset('AAPL')
df.head()

## Data quality

In [ ]:
from pprint import pprint
pprint(dmod.validate_data(df, 'AAPL'))
df['Close'].pct_change().describe(percentiles=[.01, .05, .5, .95, .99])

## Price, volume, returns and rolling volatility

In [ ]:
ret = df['Close'].pct_change().dropna()
fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=False)
df['Close'].plot(ax=axes[0])
axes[0].set_title('Adjusted close')
df['Volume'].plot(ax=axes[1], color='steelblue')
axes[1].set_title('Volume')
ret.hist(bins=100, ax=axes[2])
axes[2].set_title('Daily returns (fat tails)')
ret.rolling(20).std().plot(ax=axes[3])
axes[3].set_title('20-day rolling volatility')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'aapl_eda.png', dpi=150)
plt.show()

## Class balance and return autocorrelation

Up days are usually ~52-54%. Near-zero autocorrelation is why this problem is hard.

In [ ]:
target = (df['Close'].shift(-1) > df['Close']).astype(int)
print('Up-day share:', round(target.mean(), 4))
print('Return autocorrelations (lags 1-5):', np.round([ret.autocorr(lag=k) for k in range(1, 6)], 4))
pd.Series(target).value_counts(normalize=True).sort_index().plot.bar()
plt.title('Class balance')